In [57]:
pip install roboflow

In [58]:
import cv2
import os
from roboflow import Roboflow
from google.colab.patches import cv2_imshow # Especial para o Colab

# 1. Configurações do projeto (Extraídas do link do Roboflow)
# Obtém-se a API KEY no painel do Roboflow (Settings > Workspaces > API Keys)
rf = Roboflow(api_key="N7ccZitEfhVtcmB8EpcC")
project = rf.workspace().project("custom-workflow-object-detection-fag06")
model = project.version(4).model

loading Roboflow workspace...
loading Roboflow project...


In [59]:
# 2. Lista para as 5 fotos de cada agente carro
meus_testes = ["DECOCO.jpg","GOLARMI.jpg", "KombiCeptor.jpg", "KombusPrime.jpg", "Margheriti.jpg", "Samu.jpg", "DECOCO1.jpg","GOLARMI1.jpg", "KombiCeptor1.jpg", "KombusPrime1.jpg", "Margheriti1.jpg", "Samu1.jpg", "DECOCO2.jpg","GOLARMI2.jpg", "KombiCeptor2.jpg", "KombusPrime2.jpg", "Margheriti2.jpg", "Samu2.jpg", "carros.jpg","carros.png", "6carros.jpg", "6carros.png", "carros1.png", "carros2.png", "carros3.png", "carros4.png", "carros5.png"]

In [60]:
def processar_teste(nome_ficheiro):
    if not os.path.exists(nome_ficheiro):
        print(f"Erro: Ficheiro {nome_ficheiro} não encontrado. Faz o upload no menu lateral.")
        return

    # Carregar a imagem original com OpenCV
    img_original = cv2.imread(nome_ficheiro)
    if img_original is None:
        print(f"Erro: Não foi possível carregar a imagem {nome_ficheiro}.")
        return

    img_for_prediction = img_original.copy()
    original_height, original_width = img_for_prediction.shape[:2]

    MAX_DIM = 1024 # Define a dimensão máxima para o lado mais longo da imagem

    if max(original_height, original_width) > MAX_DIM:
        scaling_factor = MAX_DIM / float(max(original_height, original_width))
        new_width = int(original_width * scaling_factor)
        new_height = int(original_height * scaling_factor)
        img_for_prediction = cv2.resize(img_for_prediction, (new_width, new_height), interpolation=cv2.INTER_AREA)
        print(f"Imagem redimensionada de {original_width}x{original_height} para {new_width}x{new_height}")

    # Salvar a imagem (potencialmente redimensionada) num ficheiro temporário para a predição
    temp_img_path = f"/tmp/roboflow_temp_{os.path.basename(nome_ficheiro)}"
    cv2.imwrite(temp_img_path, img_for_prediction)

    # Realizar a predição
    # O Roboflow devolve as coordenadas e a classe
    prediction = model.predict(temp_img_path, confidence=40).json()

    # Eliminar o ficheiro temporário
    os.remove(temp_img_path)

    print(f"\n--- RESULTADOS: {nome_ficheiro} ---")

    # Aplicar as caixas delimitadoras na imagem que foi usada para a predição
    # Se a imagem foi redimensionada, as coordenadas da predição são para a imagem redimensionada.
    img_with_boxes = img_for_prediction.copy() # Fazer uma cópia para desenhar

    for det in prediction['predictions']:
        # Coordenadas do Roboflow (centro x, centro y, largura w, altura h)
        x, y, w, h = det['x'], det['y'], det['width'], det['height']
        classe = det['class']
        confianca = det['confidence']

        # Converter para o formato de retângulo do OpenCV (pontos superior-esquerdo e inferior-direito)
        x1 = int(x - w / 2)
        y1 = int(y - h / 2)
        x2 = int(x + w / 2)
        y2 = int(y + h / 2)

        # DESENHAR CAIXA VERDE (BGR: 0, 255, 0)
        cv2.rectangle(img_with_boxes, (x1, y1), (x2, y2), (0, 255, 0), 3)

        # COLOCAR O RÓTULO (Classe + Confiança)
        texto = f"{classe} ({confianca:.2f})"
        cv2.putText(img_with_boxes, texto, (x1, y1 - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        # IMPRIMIR COORDENADAS PARA O RELATÓRIO (como o orientador pediu)
        print(f"Objeto: {classe}")
        print(f"Centro: X={x}, Y={y}")
        print(f"Dimensões: L={w}, A={h}")
        print(f"Bounding Box: [{x1}, {y1}, {x2}, {y2}]")

    # Exibir a imagem no Colab
    cv2_imshow(img_with_boxes)

In [61]:
for foto in meus_testes:
    processar_teste(foto)

Output hidden; open in https://colab.research.google.com to view.